## Prepocessing
In this notebook, we prepare the annotated drone tiles for use in finetuning the deepforest model. The tiles are provided in TIF format while the annotations are standard COCO JSON files, one for each tile. In order to use these for training, we will need to extract the annotations and put them into two CSV files - one for training and another for validation.  

The dataset preparation involves the following steps:
1. Convert the JSON annotation files into equivalent CSV files compatible with DeepForest.
2. Merge the CSV files into a single CSV file containing all the annotations. This merged file .
3. Split these annotations in the single CSV into train/validation portions, split by image rather than rows.

In [15]:
import os
import sys

sys.path.append(os.path.abspath(".."))

import random
import pandas as pd
from glob import glob
from tqdm import tqdm
from deepforest import utilities
from scripts.coco_to_deepforest import read_coco_to_deepforest

### Step 1 - Convert JSON to CSV

In [9]:
df = read_coco_to_deepforest(
    json_path=r'D:\dsail\lacuna_field_work\Aerial\Phase_1\Annotations\2024_08_1.json', 
    output_csv_path='2024_08_1.csv', 
    root_dir='.',
    include_geometry=True
)
df.head()

,image_path,xmin,ymin,xmax,ymax,label,geometry
0,2024_08_1.png,1364,1587,1480,1707,Tree,"POLYGON ((1480 1587, 1480 1707, 1364 1707, 136..."
1,2024_08_1.png,1182,2036,1270,2120,Tree,"POLYGON ((1270 2036, 1270 2120, 1182 2120, 118..."
2,2024_08_1.png,1352,2246,1441,2345,Tree,"POLYGON ((1441 2246, 1441 2345, 1352 2345, 135..."
3,2024_08_1.png,1645,1549,1738,1641,Tree,"POLYGON ((1738 1549, 1738 1641, 1645 1641, 164..."
4,2024_08_1.png,2844,1700,2961,1816,Tree,"POLYGON ((2961 1700, 2961 1816, 2844 1816, 284..."


In [42]:
# loopthrough all JSON files and save them as CSVs
phase = "Phase1"
pattern = f"C:\\Aerial\\{phase}\\Annotations\\*.json"
out_dir = f'csv_annotations\\{phase}'
os.makedirs(out_dir, exist_ok=True)

for path in tqdm(sorted(glob(pattern))):
    out_path = os.path.join(out_dir, os.path.basename(path).split('.')[0] + '.csv')
    read_coco_to_deepforest(path, out_path, include_geometry=True)

100%|██████████| 244/244 [00:02<00:00, 98.75it/s] 


### Step 2 - Merge the CSVs

In [43]:
# merge all csv files into one
pattn = f"csv_annotations\\{phase}\\*.csv"
paths = sorted(glob(pattn))

dfs = []
for path in tqdm(paths):
    df = pd.read_csv(path)
    dfs.append(df)

merged_df = pd.concat(dfs, axis=0)
merged_df['image_path'] = merged_df['image_path'].str.replace('png', 'tif')

merged_df.to_csv('2025_merged.csv', index=False)

100%|██████████| 244/244 [00:03<00:00, 76.54it/s] 


In [44]:
utilities.read_file('2024_merged.csv', r"C:\Aerial\Phase_1\Tiles")

,image_path,xmin,ymin,xmax,ymax,label,geometry
0,2024_08_1.tif,1364,1587,1480,1707,Tree,"POLYGON ((1480 1587, 1480 1707, 1364 1707, 136..."
1,2024_08_1.tif,1182,2036,1270,2120,Tree,"POLYGON ((1270 2036, 1270 2120, 1182 2120, 118..."
2,2024_08_1.tif,1352,2246,1441,2345,Tree,"POLYGON ((1441 2246, 1441 2345, 1352 2345, 135..."
3,2024_08_1.tif,1645,1549,1738,1641,Tree,"POLYGON ((1738 1549, 1738 1641, 1645 1641, 164..."
4,2024_08_1.tif,2844,1700,2961,1816,Tree,"POLYGON ((2961 1700, 2961 1816, 2844 1816, 284..."
...,...,...,...,...,...,...,...
27201,2024_08_99.tif,974,679,1037,738,Tree,"POLYGON ((1037 679, 1037 738, 974 738, 974 679..."
27202,2024_08_99.tif,1137,1458,1272,1593,Tree,"POLYGON ((1272 1458, 1272 1593, 1137 1593, 113..."
27203,2024_08_99.tif,819,2601,903,2720,Tree,"POLYGON ((903 2601, 903 2720, 819 2720, 819 26..."
27204,2024_08_99.tif,867,3786,979,3898,Tree,"POLYGON ((979 3786, 979 3898, 867 3898, 867 37..."


### Step 3 - Split Annotations into Train/Validation/Test Sets
In this case, since the directory containing the images has tiles from only one orthophoto and the test images must be from a different orthophoto, we use a train/val split of 90/10. Afterwards, we use a test set equal in number to the validation set. 

In [45]:
# TODO: set these paths
ALL_ANNOTS_CSV = r"2024_merged.csv"   # must have: image_path,xmin,ymin,xmax,ymax,label
IMAGE_ROOT_DIR = r"D:\dsail\lacuna_field_work\Aerial\Phase_1\Tiles" # folder that contains the images referenced by image_path

OUT_DIR = r"training_dataset"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_CSV = os.path.join(OUT_DIR, "train.csv")
VAL_CSV   = os.path.join(OUT_DIR, "val.csv")

# Load and basic validation
df = pd.read_csv(ALL_ANNOTS_CSV)
required_cols = {"image_path", "xmin", "ymin", "xmax", "ymax", "label"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in {ALL_ANNOTS_CSV}: {sorted(missing)}")

# Important: avoid leakage by splitting at the image level
images = sorted(df["image_path"].dropna().unique().tolist())
if len(images) < 3:
    raise ValueError(f"Need at least 3 unique images to do a train/val split. Found {len(images)}")

random.seed(42)
random.shuffle(images)

val_frac = 0.1
n_val = max(1, int(round(len(images) * val_frac)))
val_images = set(images[:n_val])
train_images = set(images[n_val:])

train_df = df[df["image_path"].isin(train_images)].copy()
val_df   = df[df["image_path"].isin(val_images)].copy()

# Save
train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV, index=False)

print(f"Unique images: total={len(images)} train={len(train_images)} val={len(val_images)}")
print(f"Rows (boxes): train={len(train_df)} val={len(val_df)}")
print("Wrote:", TRAIN_CSV)
print("Wrote:", VAL_CSV)

Unique images: total=244 train=220 val=24
Rows (boxes): train=23918 val=3288
Wrote: training_dataset\train.csv
Wrote: training_dataset\val.csv


In [46]:
# TODO: set these paths
ALL_ANNOTS_CSV = r"2025_merged.csv"   # must have: image_path,xmin,ymin,xmax,ymax,label
IMAGE_ROOT_DIR = r"C:\Aerial\Phase_2\Tiles" # folder that contains the images referenced by image_path

OUT_DIR = r"training_dataset"
os.makedirs(OUT_DIR, exist_ok=True)

TEST_CSV = os.path.join(OUT_DIR, "test.csv")

# Load and basic validation
df = pd.read_csv(ALL_ANNOTS_CSV)
required_cols = {"image_path", "xmin", "ymin", "xmax", "ymax", "label"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in {ALL_ANNOTS_CSV}: {sorted(missing)}")

# Important: avoid leakage by splitting at the image level
images = sorted(df["image_path"].dropna().unique().tolist())
if len(images) < 3:
    raise ValueError(f"Need at least 3 unique images to do a train/val split. Found {len(images)}")

random.seed(42)
random.shuffle(images)

n_val = 30
test_images = set(images[:n_val])
test_df = df[df["image_path"].isin(test_images)].copy()

# Save
test_df.to_csv(TEST_CSV, index=False)

print(f"Unique images: total={len(images)} test={len(test_images)}")
print(f"Rows (boxes): train={len(test_df)}")
print("Wrote:", TEST_CSV)

Unique images: total=244 test=30
Rows (boxes): train=3373
Wrote: training_dataset\test.csv
